# SimBa tutorial

Companion to *How Similar Are Two Brains? A Comprehensive Benchmark of Brain
Network Similarity Measures* (Dendorfer, Luppi, Poli, Mousley, Astle and Fakhar).

- **Part A** compares two networks with the 16 measures of the paper.
- **Part B** scores your own comparison function on the benchmark and ranks it
  against all 16.

```bash
pip install git+https://github.com/ad045/simba
```

In [ ]:
import numpy as np
import pandas as pd

import simba_networks as sb

## Part A: comparing two networks

A network is a binary, symmetric (N, N) adjacency matrix without self-loops.
The package ships the 25,000 networks of the paper's generative network model
(GNM) sweep: a 50 x 50 grid over the wiring parameters $\eta$ (distance
penalty) and $\gamma$ (matching-index reward), 10 replicates per cell, 100
nodes (Schaefer-100) and 495 edges each.

In [ ]:
networks, meta = sb.load_networks()
networks.shape

Take three of them: a network `A`, another replicate `A2` from the same
parameters, and a network `B` from the opposite corner of the parameter space.

In [ ]:
cell = (meta.eta == meta.eta.unique()[22]) & (meta.gamma == meta.gamma.unique()[10])
A, A2 = networks[cell].astype(float)[:2]
B = networks[(meta.eta == meta.eta.max()) & (meta.gamma == meta.gamma.min())][0].astype(float)

Every measure is a plain function `f(A, B) -> float`. `sb.MEASURES` holds all
16, keyed by name; `sb.NAMES` gives the name used in the paper.

In [ ]:
sb.MEASURES["delta_con"](A, B)

In [ ]:
table = pd.DataFrame({
    "measure": [sb.NAMES[m] for m in sb.MEASURES],
    "similarity": [m in sb.SIMILARITIES for m in sb.MEASURES],
    "A vs A2": [f(A, A2) for f in sb.MEASURES.values()],
    "A vs B": [f(A, B) for f in sb.MEASURES.values()],
}, index=list(sb.MEASURES))
table.round(3)

Five measures are similarities (higher = more alike), the other eleven are
distances (lower = more alike). Negate the similarities before ranking or
averaging. Then every measure should place the replicate `A2` closer to `A`
than the far-off `B`:

In [ ]:
sign = np.where(table["similarity"], -1, 1)
(sign * table["A vs A2"] < sign * table["A vs B"]).all()

Your own networks go in the same way. The values depend on network size and
density, so only compare networks with the same nodes, and compare measures by
their rankings, never by their raw values.

## Part B: scoring your own measure

Write your measure as `f(A, B) -> float`, with `A` a generated network and `B`
the reference. Lower must mean more alike, or pass `similarity=True` to
`evaluate`. Return `nan` (or raise) when a comparison fails.

The example compares the two degree distributions with a Kolmogorov-Smirnov
statistic. It ignores where the edges sit, so expect it to do poorly where
topology matters.

In [ ]:
from scipy.stats import ks_2samp

def degree_ks(A, B):
    """KS statistic between the degree distributions. Lower = more alike."""
    return float(ks_2samp(A.sum(0), B.sum(0)).statistic)

degree_ks(A, A2), degree_ks(A, B)

### Accuracy: runs anywhere

Parameter recovery compares generated networks with generated networks, so it
needs no empirical data: 100 test networks at known parameters are matched to
100 grid points, once over the whole parameter space and once inside the
plausible window. The error is in grid steps (chance: 4.96 and 4.53).

`report.table` has one row per read-out, your value next to all 16 published
measures, and your `rank` among the 17 (1 = best). The two AUC rows (real
subjects against GNMs) stay empty for your measure unless you also drop in the
individual HCP connectomes.

In [ ]:
report = sb.evaluate(degree_ks, name="degree KS", criteria=["accuracy"])
report.table.drop(columns="criterion").round(2)

### The other four criteria: need the empirical reference

Agreement, plausibility, efficiency, and sensitivity and robustness compare
the 25,000 generated networks with the empirical HCP consensus connectome.
We may not redistribute it, so save your copy as
`~/simba_networks_data/reference/consensus.npy` (binary, symmetric
(100, 100), Schaefer-100 order) or pass `reference=` to `evaluate`.

Without it, the next cell falls back to a stand-in: a held-out generated
network. The code path is the same, but the numbers then describe the
stand-in, not a brain, and are not comparable to the published columns.

In [ ]:
try:
    reference = sb.load_reference()
except FileNotFoundError:
    print("No consensus found: using a generated stand-in. Numbers are NOT comparable to the paper.")
    reference = sb.load_networks("recovery_window")["targets"][0].astype(float)

report = sb.evaluate(degree_ks, name="degree KS", reference=reference,
                     criteria=["agreement", "plausibility", "efficiency"])
report.table.drop(columns="criterion").round(2)

These three criteria share one pass over the 25,000 networks.
`report.landscape` keeps your value and runtime for each of them.

The sensitivity and robustness criterion is left out above because its noise
tolerance re-scores the whole landscape up to 160 times. For the full report:

```python
report = sb.evaluate(degree_ks, n_jobs=-1)   # all five criteria, all cores
```

With `n_jobs` other than 1, define the measure in an importable module, not in
the notebook.